# Chapter 15 — Poverty and Inequality

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch15_poverty_inequality.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 15.1 What the PLFS asks about consumption

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import read_plfs, layout_names, mospi_cpi

PLFS = Path("data/plfs")
pd.DataFrame({"field": layout_names("hhv1", folder=PLFS)}).iloc[20:30].T

In [ ]:
cols = ["hh_size", "hhtype", "sg", "hce1", "hce2", "hce3", "hce4", "hce5",
        "hce_tot"]
h = read_plfs("hhv1", folder=PLFS, columns=cols)
print(f"{len(h):,} households; memory {h.memory_usage(deep=True).sum() / 1e6:.0f} MB")
h[["sec", "st", "hh_size", "hhtype", "hce1", "hce4", "hce_tot"]].head()

In [ ]:
rebuilt = h["hce1"] + h["hce2"] + h["hce3"] + (h["hce4"] + h["hce5"]) / 12
gap = (rebuilt - h["hce_tot"]).abs()
print(f"Largest difference between A+B+C+(D+E)/12 and hce_tot: Rs {gap.max():.2f}")

In [ ]:
parts = pd.Series({
    "A purchases (monthly)": h["hce1"].mul(h["weight"]).sum(),
    "B home-grown (monthly)": h["hce2"].mul(h["weight"]).sum(),
    "C wages in kind, gifts (monthly)": h["hce3"].mul(h["weight"]).sum(),
    "D clothing, footwear (annual/12)": h["hce4"].mul(h["weight"]).sum() / 12,
    "E durables (annual/12)": h["hce5"].mul(h["weight"]).sum() / 12,
})
h = h.drop(columns=["hce1", "hce2", "hce3", "hce4", "hce5"])
(100 * parts / parts.sum()).round(1)

## 15.2 From households to persons: MPCE and two kinds of weight

In [ ]:
h["mpce"] = h["hce_tot"].astype(float) / h["hh_size"].astype(float)
h["pweight"] = h["weight"] * h["hh_size"]
h["sector"] = h["sec"].map({1: "Rural", 2: "Urban"})

print(f"Households represented: {h['weight'].sum() / 1e7:.1f} crore")
print(f"Persons represented:    {h['pweight'].sum() / 1e7:.1f} crore")
print(f"Mean MPCE, household-weighted: Rs {np.average(h['mpce'], weights=h['weight']):,.0f}")
print(f"Mean MPCE, person-weighted:    Rs {np.average(h['mpce'], weights=h['pweight']):,.0f}")

In [ ]:
pers = read_plfs("perv1", folder=PLFS, columns=["age"])
print(f"Persons in the person file, weighted: {pers['weight'].sum() / 1e7:.1f} crore")
del pers

In [ ]:
size_class = h["hh_size"].clip(upper=8)
by_size = h.groupby(size_class).apply(lambda d: pd.Series({
    "share of households %": 100 * d["weight"].sum() / h["weight"].sum(),
    "share of persons %": 100 * d["pweight"].sum() / h["pweight"].sum(),
    "mean MPCE (Rs)": np.average(d["mpce"], weights=d["weight"])})).round(1)
by_size.index = [str(i) if i < 8 else "8+" for i in by_size.index]
by_size.index.name = "household size"
by_size

## 15.3 Weighted quantiles and deciles

In [ ]:
def weighted_quantile(x, w, q):
    """Value below which a share q of the total weight lies."""
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    order = np.argsort(x)
    x, w = x[order], w[order]
    cum = np.cumsum(w) / w.sum()
    return x[np.searchsorted(cum, q)]


cuts = weighted_quantile(h["mpce"], h["pweight"], np.arange(0.1, 1.0, 0.1))
pd.Series(cuts, index=[f"P{10 * k}" for k in range(1, 10)]).round(0)

In [ ]:
def add_deciles(d, w="pweight"):
    """Decile (1 = poorest tenth of persons) of each household."""
    x = d["mpce"].to_numpy(float)
    order = np.argsort(x, kind="stable")
    cum = np.cumsum(d[w].to_numpy(float)[order]) / d[w].sum()
    dec = np.empty(len(d), dtype=int)
    dec[order] = np.minimum((cum * 10 - 1e-9).astype(int) + 1, 10)
    return d.assign(decile=dec)


h = add_deciles(h)
dec = h.groupby("decile").apply(lambda d: pd.Series({
    "share of persons %": 100 * d["pweight"].sum() / h["pweight"].sum(),
    "mean MPCE (Rs)": np.average(d["mpce"], weights=d["pweight"]),
    "share of consumption %": 100 * (d["mpce"] * d["pweight"]).sum()
                              / (h["mpce"] * h["pweight"]).sum()}))
dec.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(dec.index, dec["share of consumption %"], color="#9db7d5")
ax.bar(10, dec.loc[10, "share of consumption %"], color="#1f4e79")
ax.bar(1, dec.loc[1, "share of consumption %"], color="#c55a11")
ax.axhline(10, color="grey", lw=1, ls="--")
ax.text(1.6, 10.6, "equal shares (10%)", fontsize=8, color="grey")
for k, v in dec["share of consumption %"].items():
    ax.text(k, v + 0.4, f"{v:.1f}", ha="center", fontsize=8.5)
ax.set_xticks(dec.index)
ax.set_xlabel("Decile of persons by MPCE (1 = poorest)")
ax.set_ylabel("% of total consumption")
ax.set_title("Who consumes what: shares of consumption by decile, 2023–24",
             loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.05, "Person-weighted MPCE, all India. Source: author's "
         "calculation from PLFS 2023–24 household file (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.show()

## 15.4 The Lorenz curve

In [ ]:
def lorenz(x, w):
    """Cumulative population share p and consumption share L, from (0, 0)."""
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    order = np.argsort(x)
    x, w = x[order], w[order]
    p = np.r_[0, np.cumsum(w) / w.sum()]
    L = np.r_[0, np.cumsum(w * x) / (w * x).sum()]
    return p, L


fig, ax = plt.subplots(figsize=(5.8, 5.6))
ax.plot([0, 1], [0, 1], color="grey", lw=1, ls="--", label="Perfect equality")
for s, colour in [("Rural", "#c55a11"), ("Urban", "#1f4e79")]:
    d = h[h["sector"] == s]
    p, L = lorenz(d["mpce"], d["pweight"])
    ax.plot(p, L, color=colour, lw=2, label=s)
ax.set_xlabel("Cumulative share of persons (poorest first)")
ax.set_ylabel("Cumulative share of consumption")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_title("Lorenz curves of MPCE, 2023–24", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
fig.text(0.01, -0.02, "Person-weighted. Source: author's calculation from PLFS "
         "2023–24 household file (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

## 15.5 The Gini coefficient, two ways

In [ ]:
def gini_area(x, w):
    """Gini = 1 - 2 x (area under the Lorenz curve), trapezium rule."""
    p, L = lorenz(x, w)
    area = np.sum((p[1:] - p[:-1]) * (L[1:] + L[:-1]) / 2)
    return 1 - 2 * area


def gini_cov(x, w):
    """Gini = 2 cov(x, F(x)) / mean, with F the weighted mid-point rank."""
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    order = np.argsort(x)
    x, w = x[order], w[order]
    F = (np.cumsum(w) - w / 2) / w.sum()
    mu = np.average(x, weights=w)
    cov = np.average((x - mu) * (F - np.average(F, weights=w)), weights=w)
    return 2 * cov / mu


ginis = pd.DataFrame({
    s: {"area under Lorenz": gini_area(d["mpce"], d["pweight"]),
        "covariance formula": gini_cov(d["mpce"], d["pweight"]),
        "household-weighted": gini_area(d["mpce"], d["weight"]),
        "unweighted": gini_area(d["mpce"], np.ones(len(d)))}
    for s, d in [("All India", h), ("Rural", h[h["sector"] == "Rural"]),
                 ("Urban", h[h["sector"] == "Urban"])]})
ginis.round(4)

In [ ]:
h["w_sub"] = h["mult"] / 100 / h["no_qtr"] * h["hh_size"]


def subsample_se(d, stat):
    """A statistic from each PLFS sub-sample and the half-difference SE."""
    est = {f"sub-sample {k}": stat(d.loc[d["ss"] == k, "mpce"],
                                   d.loc[d["ss"] == k, "w_sub"]) for k in (1, 2)}
    est["full sample"] = stat(d["mpce"], d["pweight"])
    est["approx. SE"] = abs(est["sub-sample 1"] - est["sub-sample 2"]) / 2
    return pd.Series(est)


pd.DataFrame({s: subsample_se(d, gini_area) for s, d in
              [("All India", h), ("Rural", h[h["sector"] == "Rural"]),
               ("Urban", h[h["sector"] == "Urban"])]}).round(4)

## 15.6 Other measures of inequality

In [ ]:
def ge(x, w, alpha):
    """Generalised entropy index GE(alpha); GE(0) = MLD, GE(1) = Theil T."""
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    r = x / np.average(x, weights=w)
    if alpha == 0:
        return np.average(-np.log(r), weights=w)
    if alpha == 1:
        return np.average(r * np.log(r), weights=w)
    return np.average(r ** alpha - 1, weights=w) / (alpha * (alpha - 1))


def palma(x, w):
    """Consumption share of the top 10% / share of the bottom 40%."""
    p, L = lorenz(x, w)
    share = lambda q: np.interp(q, p, L)
    return (1 - share(0.9)) / share(0.4)


def measures(d, w="pweight"):
    x, wt = d["mpce"], d[w]
    q = lambda k: weighted_quantile(x, wt, k)
    return pd.Series({
        "mean MPCE (Rs)": np.average(x, weights=wt), "median MPCE (Rs)": q(0.5),
        "Gini": gini_area(x, wt), "GE(0) = MLD": ge(x, wt, 0),
        "GE(1) = Theil": ge(x, wt, 1), "GE(2)": ge(x, wt, 2),
        "Palma": palma(x, wt), "P90/P10": q(0.9) / q(0.1),
        "P50/P10": q(0.5) / q(0.1), "P90/P50": q(0.9) / q(0.5)})


table = pd.DataFrame({"All India": measures(h),
                      "Rural": measures(h[h["sector"] == "Rural"]),
                      "Urban": measures(h[h["sector"] == "Urban"])})
table.round(3)

## 15.7 PLFS against the official HCES

In [ ]:
def fractile_mean(d, lo, hi, w="pweight"):
    """Mean MPCE of persons between population fractiles lo and hi."""
    x, wt = d["mpce"].to_numpy(float), d[w].to_numpy(float)
    order = np.argsort(x)
    x, wt = x[order], wt[order]
    mid = np.cumsum(wt) / wt.sum() - wt / wt.sum() / 2
    keep = (mid > lo) & (mid <= hi)
    return np.average(x[keep], weights=wt[keep])


hces = pd.DataFrame({"Rural": [4122, 0.237, 1677, 10137],
                     "Urban": [6996, 0.284, 2376, 20310]},
                    index=["mean MPCE (Rs)", "Gini", "bottom 5% mean (Rs)",
                           "top 5% mean (Rs)"])
ours = pd.DataFrame({s: [np.average(d["mpce"], weights=d["pweight"]),
                         gini_area(d["mpce"], d["pweight"]),
                         fractile_mean(d, 0, 0.05), fractile_mean(d, 0.95, 1)]
                     for s, d in h.groupby("sector")}, index=hces.index)
comp = pd.concat({"PLFS (ours)": ours, "HCES (MoSPI)": hces,
                  "PLFS / HCES": ours / hces}, axis=1)
comp.round(3)

## 15.8 States

In [ ]:
layout_file = next(PLFS.glob("Data_Layout*.xlsx"))
states = pd.read_excel(layout_file, sheet_name="State code", header=1)
states.columns = ["st", "state"]
states = states.dropna().astype({"st": int})
STATE = dict(zip(states["st"], states["state"].str.strip()))

by_state = h.groupby("st").apply(lambda d: pd.Series({
    "mean MPCE (Rs)": np.average(d["mpce"], weights=d["pweight"]),
    "Gini": gini_area(d["mpce"], d["pweight"]),
    "urban share %": 100 * d.loc[d["sec"] == 2, "pweight"].sum() / d["pweight"].sum(),
    "households in sample": len(d)}))
by_state.index = by_state.index.map(STATE)
big = by_state[by_state["households in sample"] >= 1500].sort_values("Gini")
print(f"{len(big)} states with at least 1,500 sample households")
big.round(3)

In [ ]:
nudge = {"Mizoram": (-25, -14), "Telangana": (5, -9), "Rajasthan": (-22, -13),
         "Madhya Pradesh": (-35, 8), "Jharkhand": (-48, -9),
         "Uttar Pradesh": (5, 3), "Uttarakhand": (-12, -14), "Gujarat": (6, 0),
         "West Bengal": (-20, -13), "Himachal Pradesh": (5, 0),
         "Jammu & Kashmir": (-40, 6), "Haryana": (6, 2)}
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.scatter(big["mean MPCE (Rs)"], big["Gini"], s=big["urban share %"] * 4,
           color="#9db7d5", edgecolor="#1f4e79")
for s in big.index:
    ax.annotate(s, (big.loc[s, "mean MPCE (Rs)"], big.loc[s, "Gini"]),
                fontsize=7.5, xytext=nudge.get(s, (5, 2)),
                textcoords="offset points")
ax.set_xlabel("Mean MPCE (Rs per person per month)")
ax.set_ylabel("Gini coefficient of MPCE")
ax.set_title("Average consumption and inequality across states, 2023–24",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Bubble size: urban share of persons. States with at "
         "least 1,500 sample households. Person-weighted.\nSource: author's "
         "calculation from PLFS 2023–24 household file (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.show()

In [ ]:
print(f"Correlation across these states of mean MPCE and Gini: "
      f"{big['mean MPCE (Rs)'].corr(big['Gini']):.2f}")
print(f"Correlation of urban share and Gini: "
      f"{big['urban share %'].corr(big['Gini']):.2f}")

## 15.9 How much inequality lies between groups?

In [ ]:
def theil_decomposition(d, group, w="pweight"):
    """Split Theil T and MLD into within-group and between-group parts."""
    x, wt = d["mpce"].to_numpy(float), d[w].to_numpy(float)
    mu = np.average(x, weights=wt)
    total_T, total_L = ge(x, wt, 1), ge(x, wt, 0)
    within_T = within_L = between_T = between_L = 0.0
    for _, g in d.groupby(group):
        xg, wg = g["mpce"].to_numpy(float), g[w].to_numpy(float)
        mug = np.average(xg, weights=wg)
        pop_sh = wg.sum() / wt.sum()                 # population share
        inc_sh = pop_sh * mug / mu                   # consumption share
        within_T += inc_sh * ge(xg, wg, 1)
        between_T += inc_sh * np.log(mug / mu)
        within_L += pop_sh * ge(xg, wg, 0)
        between_L += pop_sh * np.log(mu / mug)
    return pd.DataFrame({
        "Theil T": [total_T, within_T, between_T, 100 * between_T / total_T],
        "MLD": [total_L, within_L, between_L, 100 * between_L / total_L]},
        index=["total", "within groups", "between groups", "between, % of total"])


h["state_sector"] = h["st"].astype(str) + "-" + h["sec"].astype(str)
decomp = pd.concat({"sector": theil_decomposition(h, "sector"),
                    "state": theil_decomposition(h, "st"),
                    "state × sector": theil_decomposition(h, "state_sector")},
                   axis=1)
decomp.round(3)

In [ ]:
SG = {1: "Scheduled Tribe", 2: "Scheduled Caste", 3: "OBC", 9: "Others"}
h["social_group"] = h["sg"].map(SG)
decomp_sg = theil_decomposition(h, "social_group")
print(f"Between social groups: {decomp_sg.loc['between, % of total', 'Theil T']:.1f}% "
      f"of Theil T, {decomp_sg.loc['between, % of total', 'MLD']:.1f}% of MLD")

## 15.10 Poverty: the FGT family

### Choosing and updating a line

In [ ]:
deflator = pd.Series({"Rural": 4122 / 2079, "Urban": 6996 / 3632})
line_2011 = pd.Series({"Rural": 972, "Urban": 1407})      # Rangarajan, 2011-12
line_2023 = line_2011 * deflator

cpi = {}
for s in ("rural", "urban"):
    c = mospi_cpi(s)
    year = c[(c["date"] >= "2023-07-01") & (c["date"] <= "2024-06-01")]
    cpi[s.title()] = year["index"].mean()
cpi = pd.Series(cpi)

pd.DataFrame({"line 2011-12 (Rs)": line_2011,
              "HCES implicit deflator": deflator,
              "line 2023-24 (Rs)": line_2023,
              "CPI Jul 2023-Jun 2024 (2012 = 100)": cpi,
              "line via CPI only (Rs)": line_2011 * cpi / 100}).round(2)

In [ ]:
def fgt(x, w, z, alpha):
    """Foster-Greer-Thorbecke index: mean of ((z - x)/z)^alpha among the poor."""
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    gap = np.clip((z - x) / z, 0, None)
    return np.average(np.where(x < z, gap ** alpha, 0.0), weights=w)


def poverty(d, z):
    return pd.Series({"headcount P0 %": 100 * fgt(d["mpce"], d["pweight"], z, 0),
                      "poverty gap P1 %": 100 * fgt(d["mpce"], d["pweight"], z, 1),
                      "squared gap P2 %": 100 * fgt(d["mpce"], d["pweight"], z, 2)})


med = {s: weighted_quantile(d["mpce"], d["pweight"], 0.5)
       for s, d in h.groupby("sector")}
scale = ours.loc["mean MPCE (Rs)"] / hces.loc["mean MPCE (Rs)"]
lines = {s: {"1 Rangarajan updated": line_2023[s],
             "2 Rangarajan x PLFS/HCES": line_2023[s] * scale[s],
             "3 60% of median": 0.6 * med[s]} for s in ("Rural", "Urban")}
pov = pd.DataFrame({(s, name): pd.concat([pd.Series({"line (Rs)": z}),
                                          poverty(h[h["sector"] == s], z)])
                    for s in lines for name, z in lines[s].items()}).T
pov.round(2)

In [ ]:
z_grid = np.linspace(800, 4000, 161)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, s in zip(axes, ["Rural", "Urban"]):
    d = h[h["sector"] == s]
    for a, colour, lab in [(0, "#1f4e79", "P0 headcount"),
                           (1, "#c55a11", "P1 poverty gap"),
                           (2, "#9db7d5", "P2 squared gap")]:
        ax.plot(z_grid, [100 * fgt(d["mpce"], d["pweight"], z, a) for z in z_grid],
                color=colour, lw=2, label=lab)
    for name, ls in [("1 Rangarajan updated", "--"), ("3 60% of median", ":")]:
        z = lines[s][name]
        ax.axvline(z, color="black", lw=1, ls=ls)
        ax.text(z + 40, 92 if ls == "--" else 74, f"line {name[0]}", fontsize=8)
    ax.set_title(s, loc="left", fontsize=12)
    ax.set_xlabel("Poverty line, Rs per person per month")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("% of persons (P0) / % of the line (P1, P2)")
axes[0].set_ylim(0, 100)
axes[1].legend(frameon=False, loc="upper left")
fig.suptitle("How the FGT measures change with the poverty line, 2023–24",
             x=0.01, ha="left", fontsize=13)
fig.text(0.01, -0.04, "Lines: 1 = Rangarajan (2014) line updated with MoSPI's "
         "HCES deflator; 3 = 60% of the sector median. Person-weighted MPCE "
         "from PLFS 2023–24\nhousehold file (NSO, MoSPI); author's calculation.",
         fontsize=8, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
ten = {s: (poverty(d, line_2023[s] * 1.1) - poverty(d, line_2023[s])).round(2)
       for s, d in h.groupby("sector")}
pd.DataFrame(ten).rename_axis("change when line 1 rises by 10%")

In [ ]:
rel = h.groupby(["sector", "social_group"]).apply(
    lambda d: poverty(d, lines[d.name[0]]["3 60% of median"]))
rel["share of persons %"] = (h.groupby(["sector", "social_group"])["pweight"].sum()
                             / h.groupby("sector")["pweight"].sum() * 100)
rel.round(1)

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading